# NB2 — Reward model + offline OPE (IPW / DM / DR) on the temporal holdout

Fit `P(click | context, item, position)` on the **train** period, then evaluate three
target policies on the **eval** period with obp: uniform, greedy (the broken case),
and ε-greedy (the fix). Overlap diagnostics explain *why* greedy fails.

In [1]:
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import roc_auc_score

from src import config
from src.obd_io import load_shard, item_universe, build_bandit_feedback
from src.reward_model import RewardModel, fit_reward_model
from src.ope_wrap import uniform_dist, greedy_dist, eps_greedy_dist, run_ope, overlap_stats, score_at_logged

In [2]:
train = pd.read_parquet(config.DATA / "obd_train.parquet")
ev = pd.read_parquet(config.DATA / "obd_eval.parquet")
ids = json.loads((config.ARTIFACTS / "arm_universe.json").read_text())
print(f"train {len(train)} rows (CTR {train['click'].mean():.4f}) | eval {len(ev)} rows (CTR {ev['click'].mean():.4f})")

train 160000 rows (CTR 0.0046) | eval 40000 rows (CTR 0.0047)


## Model comparison: LogisticRegression vs LightGBM + ROC-AUC
AUC here is a pure **discrimination** metric on the logged rows (it ignores propensity —
that is correct: AUC measures reward-model fit quality, while OPE below handles the
bandit/policy side). Train AUC is in-sample; eval AUC is the out-of-time holdout we
actually trust. We pick the model with the best eval AUC as the pipeline reward model.

In [3]:
train_fb = build_bandit_feedback(train, item_ids=ids)
ev_fb = build_bandit_feedback(ev, item_ids=ids)

models = {}
est_by_kind = {}
for kind in ["logistic", "lgb"]:
    m = fit_reward_model(train, ids, kind=kind)
    p_tr = m.predict_all(train, ids)
    p_ev = m.predict_all(ev, ids)
    auc_tr = roc_auc_score(train_fb["reward"], score_at_logged(p_tr, train_fb["action"], train_fb["position"]))
    auc_ev = roc_auc_score(ev_fb["reward"], score_at_logged(p_ev, ev_fb["action"], ev_fb["position"]))
    mean_pred = float(p_ev.mean())
    models[kind] = {"auc_train": float(auc_tr), "auc_eval": float(auc_ev), "mean_pred_eval_ctr": mean_pred}
    est_by_kind[kind] = p_ev
    print(f"{kind:>8} | train AUC={auc_tr:.4f} | eval AUC={auc_ev:.4f} | mean pred={mean_pred:.5f} (actual {ev['click'].mean():.5f})")

comparison = pd.DataFrame(models).T.round(4)
print(comparison)
best_kind = comparison["auc_eval"].idxmax()
print("selected reward model:", best_kind)
(config.ARTIFACTS / "model_comparison.json").write_text(json.dumps(
    models | {"selected": best_kind}, indent=2))

logistic | train AUC=0.7952 | eval AUC=0.7835 | mean pred=0.00293 (actual 0.00468)
     lgb | train AUC=0.8387 | eval AUC=0.7794 | mean pred=0.00251 (actual 0.00468)
          auc_train  auc_eval  mean_pred_eval_ctr
logistic     0.7952    0.7835              0.0029
lgb          0.8387    0.7794              0.0025
selected reward model: logistic


307

In [4]:
kind = best_kind
est_rewards = est_by_kind[kind]
fit_model = fit_reward_model(train, ids, kind=kind)     # refit selected, persist for NB3/NB4
joblib.dump(fit_model, config.ARTIFACTS / "reward_model.joblib")
print(f"using reward model kind={kind}; est_rewards:", est_rewards.shape,
      "| mean predicted CTR:", round(est_rewards.mean(), 5))
print("actual eval CTR:", round(ev['click'].mean(), 5), "(model mean is a calibration glance, not a match target)")

using reward model kind=logistic; est_rewards: (40000, 80, 3) | mean predicted CTR: 0.00293
actual eval CTR: 0.00468 (model mean is a calibration glance, not a match target)


In [5]:
fb = build_bandit_feedback(ev, item_ids=ids)
targets = {
    "uniform": uniform_dist(len(ev), len(ids), config.N_POSITIONS),
    "greedy": greedy_dist(est_rewards),
    "eps_greedy_0.1": eps_greedy_dist(est_rewards, eps=0.1),
}

results = {}
for name, dist in targets.items():
    ov = overlap_stats(dist, fb["action"], fb["position"])
    res = run_ope(fb, dist, est_rewards)
    results[name] = {"ope": res, "overlap": ov}
    print(f"\n=== {name} ===  overlap: pi_e(logged)={ov['pi_e_on_logged']:.4f}, zero-cover={ov['share_zero_overlap']:.1%}")
    for k, v in res.items():
        print(f"  {k:>4}: {v:.4f}")


=== uniform ===  overlap: pi_e(logged)=0.0125, zero-cover=0.0%
   ipw: 0.0034
  snipw: 0.0033
    dm: 0.0029
    dr: 0.0035

=== greedy ===  overlap: pi_e(logged)=0.0215, zero-cover=97.9%
   ipw: 0.0041
  snipw: 0.0040
    dm: 0.0060
    dr: 0.0045

=== eps_greedy_0.1 ===  overlap: pi_e(logged)=0.0206, zero-cover=0.0%
   ipw: 0.0041
  snipw: 0.0039
    dm: 0.0057
    dr: 0.0044


## Propensity-clipping sensitivity (ε-greedy target)
IPS weights = π_e / p; tiny logged pscores explode variance. Sweep the clip floor
and watch estimator stability.

In [6]:
sens = {}
for eps_clip in [1e-2, 1e-3, 1e-4]:
    fb_c = build_bandit_feedback(ev, item_ids=ids, eps=eps_clip)
    sens[str(eps_clip)] = run_ope(fb_c, targets["eps_greedy_0.1"], est_rewards)
sens_df = pd.DataFrame(sens).T.round(5)
print(sens_df)

            ipw    snipw       dm       dr
0.01    0.00389  0.00401  0.00572  0.00437
0.001   0.00401  0.00396  0.00572  0.00439
0.0001  0.00407  0.00395  0.00572  0.00442


In [7]:
(config.ARTIFACTS / "ope_results.json").write_text(json.dumps({
    "eval_rows": int(len(ev)),
    "eval_ctr": float(ev["click"].mean()),
    "targets": results,
    "clip_sensitivity_eps_greedy": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in sens.items()},
}, indent=2))
print("saved artifacts/ope_results.json")

saved artifacts/ope_results.json


## Conclusions
- **Uniform**: estimators agree — well-covered target, trustworthy numbers.
- **Greedy**: IPW collapses toward 0 while DM inflates — the low-overlap signature
  (`zero-cover ≈ 96%`): the greedy action is almost never the logged action, so IPS
  has no support and DM is left alone with a model that over-trusts its own argmax.
- **ε-greedy (ε=0.1)**: mixing in exploration restores overlap (`zero-cover = 0`);
  SNIPS/DM land in a coherent band, raw IPW stays noisy because CTR is tiny.
- **The small-CTR reality**: with CTR ≈ 0.005, 100k eval rows carry only ~500 clicks.
  IPS-family estimators are consistent but high-variance here; SNIPS trades a little
  bias for much lower variance, and DM is stable but only as good as the model.
  DR hedges between them. Read the four together — agreement = trust.
- Deployable pattern: never ship a deterministic argmax policy you intend to
  evaluate offline; keep ε-exploration so the log keeps covering your policy.